# Unified retrieval: smooth half-cut interiors

The primary comparison uses the original ground-truth triangles, the decoded Objective 1 triangles from conditioning view 18, and the category-blind unified retrieval triangles. Every category uses the same component and structural hypotheses, shared selector, and global confidence gate. All meshes use the same cut plane, camera, lighting, and material. The manifest contains the strongest available examples per category and explicitly reports fallbacks.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pyvista as pv
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/retrieval_unified_view18")
OBJECTIVE1_MESH_DIR = Path("results/objective1_view18_full/predictions/objective1/seed_42/mesh")
GROUND_TRUTH_MESH_DIR = RESULTS_DIR / "references" / "ground_truth_mesh"
FINAL_METHOD = "retrieval_unified"
RETRIEVAL_MESH_DIR = RESULTS_DIR / "predictions" / FINAL_METHOD / "mesh"
RESOLUTION = 64
METRIC_MARGIN = 2
CATEGORIES = ("bus", "cabinet", "car", "file_cabinet")

# Remove the positive-x half to open one side of every consistently normalized shape.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_FRACTION = 0.50
METHODS = [
    ("Ground truth", GROUND_TRUTH_MESH_DIR, "#d9dde5"),
    ("Objective 1 · view 18", OBJECTIVE1_MESH_DIR, "#f4a261"),
    ("Unified retrieval", RETRIEVAL_MESH_DIR, "#55bde8"),
]

for required in (RESULTS_DIR / "per_sample.csv", RESULTS_DIR / "visualization_manifest.csv"):
    if not required.is_file():
        raise FileNotFoundError(f"Missing result file: {required}")

metrics = {}
with (RESULTS_DIR / "per_sample.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if row["category"] in CATEGORIES and int(row["margin"]) == METRIC_MARGIN:
            metrics.setdefault(row["sample_id"], {})[row["method"]] = {
                "category": row["category"],
                "precision": float(row["internal_precision"]),
                "recall": float(row["internal_recall"]),
                "f1": float(row["internal_f1"]),
                "ratio": float(row["pred_to_gt_internal_ratio"]),
                "components": int(row["internal_components_26"]),
                "core": float(row["volumetric_core_fraction"]),
                "selected_rank": int(row["selected_rank"]),
                "fallback": bool(int(row["used_objective1_fallback"])),
                "gate_reason": row["gate_reason"],
                "policy_mode": row.get("policy_mode", "v2_hybrid"),
            }

sample_ids = []
gallery_categories = []
with (RESULTS_DIR / "visualization_manifest.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if row["category"] in CATEGORIES:
            sample_ids.append(row["sample_id"])
            gallery_categories.append(row["category"])
if not sample_ids:
    raise ValueError("Visualization manifest contains no requested samples")
if len(sample_ids) != len(set(sample_ids)):
    raise ValueError("Visualization manifest contains duplicate sample IDs")
category_counts = {category: gallery_categories.count(category) for category in CATEGORIES}
missing_categories = [category for category, count in category_counts.items() if count == 0]
if missing_categories:
    raise ValueError(f"Manifest has no samples for: {missing_categories}")
for sample_id in sample_ids:
    if set(metrics.get(sample_id, {})) != {"objective1", FINAL_METHOD}:
        raise ValueError(f"Incomplete metrics for {sample_id}")
    for _, directory, _ in METHODS:
        path = directory / f"{sample_id}.ply"
        if not path.is_file():
            raise FileNotFoundError(f"Missing smooth mesh: {path}")

def f1_delta(sample_id):
    return metrics[sample_id][FINAL_METHOD]["f1"] - metrics[sample_id]["objective1"]["f1"]

print(f"Verified {len(sample_ids)} smooth unified cutaways: {category_counts}")
for index, sample_id in enumerate(sample_ids):
    final = metrics[sample_id][FINAL_METHOD]
    state = "fallback" if final["fallback"] else f"{final['policy_mode']} · donor rank {final['selected_rank']}"
    print(f"  {index:02d}: {final['category']:3s} · ΔF1 {f1_delta(sample_id):+.3f} · {state} · {sample_id}")

In [ ]:
def read_triangle_surface(path):
    mesh = pv.read(path)
    if isinstance(mesh, pv.MultiBlock):
        mesh = mesh.combine()
    surface = mesh.extract_surface().triangulate().clean()
    if surface.n_cells == 0:
        raise ValueError(f"Empty triangle surface: {path}")
    return surface


def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([1.0, 0.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.1 * normal + 0.45 * side + 0.30 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def render_smooth_comparison(sample_id):
    surfaces = {label: read_triangle_surface(directory / f"{sample_id}.ply") for label, directory, _ in METHODS}
    ground_truth = surfaces["Ground truth"]
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    projection = np.asarray(ground_truth.points) @ normal
    cut_offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    cut_origin = np.asarray(ground_truth.center) + normal * (cut_offset - np.asarray(ground_truth.center) @ normal)
    center = np.asarray(ground_truth.center)
    object_size = ground_truth.length

    plotter = pv.Plotter(shape=(1, 3), off_screen=True, window_size=(1800, 620), border=False)
    try:
        plotter.enable_anti_aliasing("ssaa")
    except Exception:
        pass
    for column, (label, _, color) in enumerate(METHODS):
        cut = surfaces[label].clip(normal=normal, origin=cut_origin, invert=True)
        plotter.subplot(0, column)
        plotter.set_background("white")
        plotter.add_mesh(cut, color=color, smooth_shading=True, ambient=0.30, diffuse=0.72, specular=0.18, show_edges=False)
        plotter.add_text(label, position="upper_left", color="black", font_size=11)
    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.56 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)

In [ ]:
SAMPLE_INDEX = 0  # See the verified manifest listing above.

sample_id = sample_ids[SAMPLE_INDEX]
baseline = metrics[sample_id]["objective1"]
final = metrics[sample_id][FINAL_METHOD]
print(f"{sample_id} · {final['category']}")
print(f"Internal F1: {baseline['f1']:.3f} → {final['f1']:.3f} ({f1_delta(sample_id):+.3f})")
print(f"Precision: {baseline['precision']:.3f} → {final['precision']:.3f} · Recall: {baseline['recall']:.3f} → {final['recall']:.3f}")
print(f"Internal ratio: {baseline['ratio']:.2f} → {final['ratio']:.2f} · components: {baseline['components']} → {final['components']}")
print(f"Policy: {final['policy_mode']} · gate: {final['gate_reason']} · selected DINO rank: {final['selected_rank']}")
display(render_smooth_comparison(sample_id))

In [ ]:
SHOW_ALL_BEST = True

if SHOW_ALL_BEST:
    for index, sample_id in enumerate(sample_ids):
        baseline = metrics[sample_id]["objective1"]
        final = metrics[sample_id][FINAL_METHOD]
        print(f"{index:02d}. {final['category']} · F1 {baseline['f1']:.3f} → {final['f1']:.3f} ({f1_delta(sample_id):+.3f}) · {sample_id}")
        display(render_smooth_comparison(sample_id))

## Optional fair voxel diagnostic

The next cell reconstructs every method from its 64³ voxel surface. Use it only when you want a representation-matched check that also includes the old coherent-retrieval v1 result.

In [ ]:
SHOW_V1_VOXEL_DIAGNOSTIC = False
V1_DIR = Path("results/coherent_retrieval_view18/predictions/coherent_retrieval")
GT_VOXEL_DIR = RESULTS_DIR / "references" / "ground_truth_voxels"
OBJECTIVE1_VOXEL_DIR = Path("results/objective1_view18_full/predictions/objective1/seed_42/voxels")
V21_VOXEL_DIR = RESULTS_DIR / "predictions" / FINAL_METHOD / "voxels"

def voxel_surface(path):
    points = np.asarray(pv.read(path).points, dtype=np.float32).reshape(-1, 3)
    coordinates = np.unique(np.clip(np.floor((points + 0.5) * RESOLUTION).astype(np.int32), 0, RESOLUTION - 1), axis=0)
    occupancy = np.zeros((RESOLUTION, RESOLUTION, RESOLUTION), dtype=np.float32)
    occupancy[tuple(coordinates.T)] = 1.0
    field = np.pad(occupancy, 1)
    spacing = 1.0 / RESOLUTION
    grid = pv.ImageData(dimensions=field.shape, spacing=(spacing,) * 3, origin=(-0.5 - 0.5 * spacing,) * 3)
    grid.point_data["occupancy"] = field.ravel(order="F")
    return grid.contour([0.5], scalars="occupancy").clean().triangulate().smooth(n_iter=30, relaxation_factor=0.04, boundary_smoothing=False)

def render_voxel_diagnostic(sample_id):
    sources = [
        ("Ground truth", GT_VOXEL_DIR, "#d9dde5"),
        ("Objective 1", OBJECTIVE1_VOXEL_DIR, "#f4a261"),
        ("Retrieval v1", V1_DIR, "#9b8ac4"),
        ("Unified retrieval", V21_VOXEL_DIR, "#55bde8"),
    ]
    surfaces = {label: voxel_surface(directory / f"{sample_id}.ply") for label, directory, _ in sources}
    gt = surfaces["Ground truth"]
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    projection = np.asarray(gt.points) @ normal
    offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    origin = np.asarray(gt.center) + normal * (offset - np.asarray(gt.center) @ normal)
    plotter = pv.Plotter(shape=(1, 4), off_screen=True, window_size=(2200, 600), border=False)
    for column, (label, _, color) in enumerate(sources):
        plotter.subplot(0, column)
        plotter.set_background("white")
        plotter.add_mesh(surfaces[label].clip(normal=normal, origin=origin, invert=True), color=color, smooth_shading=True, show_edges=False)
        plotter.add_text(label, position="upper_left", color="black", font_size=10)
    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, np.asarray(gt.center), gt.length)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.56 * gt.length
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)

if SHOW_V1_VOXEL_DIAGNOSTIC:
    display(render_voxel_diagnostic(sample_ids[SAMPLE_INDEX]))